# Meep: 2D Directional Coupler

Meep remains supported for capabilities not yet available in GDSFactory FDTD.
This notebook is the maintained reference for a true 2D effective-index simulation.

**Requirements:** [GDSFactory+](https://gdsfactory.com) cloud simulation access.

### Load a pcell from UBC PDK

In [ ]:
import gdsfactory as gf

gf.gpdk.PDK.activate()

c = gf.components.coupler(gap=0.5)
c

### Configure 2D simulation

The only difference from a 3D simulation is `sim.solver(mode="2d", z_cut="auto")`.
This collapses the z-dimension, ignores sidewall angles, and enforces TE polarization.

In [ ]:
from gsim import meep
from gsim.common.stack import get_stack
from gsim.meep.models.api import Material

stack = get_stack()  # auto-detects active PDK

sim = meep.Simulation()

sim.geometry(component=c, stack=stack)
sim.materials = {
    "si": Material(refractive_index=3.47),
    "SiO2": Material(refractive_index=1.44),
}
sim.source(port="o1", wavelength=1.55, wavelength_span=0.01)
sim.monitors = ["o1", "o2", "o3"]
sim.domain(pml=1.0, margin_x=0.5, margin_y=0.5)
sim.solver(resolution=25, mode="2d", z_cut="auto")
sim.num_freqs = 21
sim.solver.stop_when_energy_decayed()

print(sim.validate_config())

### Preview geometry

In [ ]:
sim.plot_2d(slices="z")

For an interactive preview, use `plot_2d_interactive()`. It returns a Plotly
figure where you can zoom, pan, and toggle individual layers, materials, PML
regions, and ports on/off via the legend.

In [ ]:
sim.plot_2d_interactive()

### Run 2D simulation on cloud

In [ ]:
result = sim.run(check_cache=True)

In [ ]:
result.plot_interactive()

In [ ]:
result.plot_interactive(phase=True)